# VibeML: Evaluation Traces

This notebook runs the VibeML agent through five trace scenarios and evaluates the results using MLflow. One trace runs the same scenario through two different LLMs to compare performance. Two traces test how the agent handles requests that fall outside its scope. A custom judge is used to score the agent's recommendations, and commentary is provided on what the evaluation showed.

## Setup

This pulls in the tools and agent logic built in the previous notebook and sets up an MLflow experiment to track all five traces.


In [0]:
%pip install --upgrade "mlflow[databricks]>=3.4.0"
dbutils.library.restartPython()

In [0]:
import mlflow
import json
import pandas as pd

# Set up the MLflow experiment for this notebook
mlflow.set_experiment("/Shared/VibeML_Evaluation_Traces")

print("MLflow experiment set")

## Bringing in the Agent

This pulls in everything from the agent prototype notebook, including the tools, the Spotify connection, the LLM setup, and the datasets, so we can run the agent here without rebuilding it.

In [0]:
%run "./02_agent_prototype"

## MLflow Tracing

Each trace scenario is wrapped in @mlflow.trace so it gets a real trace_id, viewable in the MLflow Traces UI. The judge is built using make_judge(), and both the judge's score and the human review are logged as Feedback directly on each trace.

#### Shared Setup for Trace Scenarios

This section wraps all 8 tools with tracing, defines all 5 user requests used across the trace scenarios, and defines run_rejection_trace, the shared function used by Trace 4 and 5. run_vibeml_trace (used by Trace 1, 2, and 3) is defined separately right after this.

In [0]:
import mlflow

# Wrap each tool so tool calls appear as spans inside each request trace
get_user_profile = mlflow.trace(name="get_user_profile")(get_user_profile)
search_songs = mlflow.trace(name="search_songs")(search_songs)
vibe_mapping = mlflow.trace(name="vibe_mapping")(vibe_mapping)
feedback_refinement = mlflow.trace(name="feedback_refinement")(feedback_refinement)
constraint_filter = mlflow.trace(name="constraint_filter")(constraint_filter)
explicit_content_filter = mlflow.trace(name="explicit_content_filter")(explicit_content_filter)
deduplication = mlflow.trace(name="deduplication")(deduplication)
search_catalog = mlflow.trace(name="search_catalog")(search_catalog)
liked_songs_matcher = mlflow.trace(name="liked_songs_matcher")(liked_songs_matcher)
get_currently_playing = mlflow.trace(name="get_currently_playing")(get_currently_playing)
trend_filter = mlflow.trace(name="trend_filter")(trend_filter)
generate_playlist_title_description = mlflow.trace(name="generate_playlist_title_description")(generate_playlist_title_description)
save_playlist = mlflow.trace(name="save_playlist")(save_playlist)

print("All 13 tools wrapped with tracing")

In [0]:
user_request_1 = "I want fast afrobeat music for the gym"
user_request_2 = "I'm going on a late night drive, give me something that matches that vibe"
user_request_3 = "I need to focus, I'm studying for an exam. Something calm with no lyrics please"
user_request_4 = "Can you recommend a good podcast to listen to on my commute?"
user_request_5 = "Hey, what stocks should I invest in right now?"

print("All 5 user requests defined")

In [0]:
@mlflow.trace(name="vibeml_recommendation")
def run_vibeml_trace(user_request, llm_model_override=None, include_save=False, era=None, popularity_range=None, check_currently_playing=False):
    """
    Runs one full recommendation cycle.
    
    Args:
        user_request (str): The user's request
        llm_model_override (str): Optionally swap the LLM for this trace
        include_save (bool): If True, generates a title/description and attempts to save
        era (tuple): Optional (start_year, end_year), triggers trend_filter
        popularity_range (tuple): Optional (min, max) popularity, triggers trend_filter
        check_currently_playing (bool): If True, also calls get_currently_playing for context
    """
    global LLM_MODEL
    original_model = LLM_MODEL
    if llm_model_override:
        LLM_MODEL = llm_model_override

    output = {}

    if check_currently_playing:
        output["currently_playing"] = get_currently_playing(sp)

    ranges = vibe_mapping(user_request, profile)
    explicit_genre = extract_explicit_genre(user_request)
    if explicit_genre:
        ranges["genre"] = explicit_genre

    results = search_songs(
        energy=(ranges["energy"]["min"], ranges["energy"]["max"]),
        valence=(ranges["valence"]["min"], ranges["valence"]["max"]),
        danceability=(ranges["danceability"]["min"], ranges["danceability"]["max"]),
        genre=ranges.get("genre"),
        limit=20,
        profile=profile,
        df_tracks=df_tracks,
        df_popularity=df_popularity
    )
    results = constraint_filter(results, explicit=False, min_popularity=30)

    if era or popularity_range:
        results = trend_filter(results, popularity=popularity_range, era=era)

    results = deduplication(results, max_per_artist=1)

    liked_matches = liked_songs_matcher(profile, ranges, df_tracks)
    if len(liked_matches) > 0:
        liked_matches["is_familiar"] = True
        results = pd.concat([liked_matches, results]).drop_duplicates(subset=["track_name"]).reset_index(drop=True)

    if llm_model_override:
        LLM_MODEL = original_model

    top_results = results.head(5)

    verified_songs = []
    for _, row in top_results.iterrows():
        catalog_result = search_catalog(sp, track_name=row["track_name"], artist_name=row["artists"].split(";")[0])
        verified_songs.append({
            "track_name": row["track_name"],
            "artists": row["artists"],
            "track_genre": row["track_genre"],
            "verified_on_spotify": catalog_result is not None,
            "track_id": catalog_result["track_id"] if catalog_result else None
        })

    output["genre"] = ranges.get("genre")
    output["songs_found"] = len(results)
    output["top_songs"] = verified_songs

    if include_save:
        sample_for_title = [{"track_name": s["track_name"], "artist": s["artists"].split(";")[0]} for s in verified_songs]
        playlist_meta = generate_playlist_title_description(user_request, sample_tracks=sample_for_title, profile=profile)

        track_ids = [s["track_id"] for s in verified_songs if s["track_id"]]
        save_result = save_playlist(sp, track_ids, playlist_meta["title"], playlist_meta["description"])

        output["playlist_title"] = playlist_meta["title"]
        output["playlist_description"] = playlist_meta["description"]
        output["save_result"] = save_result

    return output

In [0]:
trace1_full_result = run_vibeml_trace(user_request_1, llm_model_override="databricks-gpt-oss-120b", include_save=True)
trace1_full_id = mlflow.get_last_active_trace_id()

print(f"Trace ID: {trace1_full_id}")
print(f"Playlist title: {trace1_full_result['playlist_title']}")
print(f"Description: {trace1_full_result['playlist_description']}")
print(f"Save result: {json.dumps(trace1_full_result['save_result'], indent=2)}")

In [0]:
@mlflow.trace(name="vibeml_rejection")
def run_rejection_trace(user_request):
    """
    Tests whether the agent gracefully declines an out-of-scope request.
    """
    rejection_prompt = f"""You are VibeML, a personalized Spotify playlist agent. You only help with music playlist requests.
    
    A user just said: "{user_request}"
    
    Respond naturally, explaining that this falls outside what you can help with since you are scoped to music playlists, 
    and then redirect them back to building a playlist."""
    
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": rejection_prompt}]
    )
    
    content = response.choices[0].message.content
    if isinstance(content, list):
        rejection_text = " ".join([block["text"] for block in content if block.get("type") == "text"])
    else:
        rejection_text = content
    
    return rejection_text

#### Separate Trace: User Profile Pull

get_user_profile only runs once per session in the real agent, not once per request, so it gets its own standalone trace rather than being inside every scenario.

In [0]:
@mlflow.trace(name="vibeml_profile_pull")
def run_profile_trace():
    return get_user_profile(sp)

profile_result = run_profile_trace()
profile_trace_id = mlflow.get_last_active_trace_id()

print(f"Profile trace logged: {profile_trace_id}")
print(f"Top artists: {profile_result['top_artists'][:5]}")
print(f"Preferred genres: {profile_result['preferred_genres']}")

##### Commentary


#### Trace 1: Gym / High Energy (Dual LLM Comparison)

This trace runs the same user request through two different LLMs, databricks-gpt-oss-120b and databricks-meta-llama-3-3-70b-instruct, so we can compare how each one interprets the mood and picks songs.

In [0]:
trace1_result = run_vibeml_trace(user_request_1, llm_model_override="databricks-gpt-oss-120b")
trace1_id = mlflow.get_last_active_trace_id()

print(f"Trace ID: {trace1_id}")
print(f"Genre: {trace1_result['genre']}")
print(f"Songs found: {trace1_result['songs_found']}")
for song in trace1_result["top_songs"]:
    print(f"  {song['track_name']} by {song['artists']} (verified on Spotify: {song['verified_on_spotify']})")

In [0]:
trace1b_result = run_vibeml_trace(user_request_1, llm_model_override="databricks-meta-llama-3-3-70b-instruct")
trace1b_id = mlflow.get_last_active_trace_id()

print(f"Trace ID: {trace1b_id}")
print(f"Genre: {trace1b_result['genre']}")
print(f"Songs found: {trace1b_result['songs_found']}")
for song in trace1b_result["top_songs"]:
    print(f"  {song['track_name']} by {song['artists']} (verified on Spotify: {song['verified_on_spotify']})")

##### Commentary
The two models diverged instead of agreeing. GPT-OSS-120B returned 5 songs, but two of them, Enzo by Plastilina Mosh and Fellini by Criolo, are Latin rock and Brazilian rap, not afrobeat at all. The judge caught this and scored it a 2. Llama 3.3 70B returned 3 songs, all genuinely tagged afrobeat, and scored a 3. Taken together, the two runs suggest the models are not consistently equivalent, sometimes they agree, sometimes one drifts into a genre mismatch the other avoids, which means model choice does matter here, just not in a way that's predictable from a single test.

#### Trace 2: Late Night Drive

This trace tests a more abstract, mood-based request rather than a direct genre request. The user describes a feeling rather than naming a genre, so this checks how well the agent translates context into audio features without an explicit genre override.

In [0]:
trace2_result = run_vibeml_trace(user_request_2)
trace2_id = mlflow.get_last_active_trace_id()

print(f"Trace ID: {trace2_id}")
print(f"Genre: {trace2_result['genre']}")
print(f"Songs found: {trace2_result['songs_found']}")
for song in trace2_result["top_songs"]:
    print(f"  {song['track_name']} by {song['artists']} (verified on Spotify: {song['verified_on_spotify']})")

##### Commentary
This was the strongest trace. With no explicit genre given, the agent correctly inferred deep-house from a mood description alone, and the results back it up: 15 songs, led by a track literally called "Drive." This shows the vibe mapping tool isn't just keyword matching, it's making a real judgment call about what audio features suit "late night drive," and getting it right.

#### Trace 3: Studying / No Lyrics

This trace tests a functional request rather than a mood-based one. The user wants something specific for focus, low energy, no lyrics, which checks whether the agent correctly prioritizes low speechiness and high instrumentalness instead of just defaulting to a generic "calm" genre.

In [0]:
trace3_result = run_vibeml_trace(user_request_3)
trace3_id = mlflow.get_last_active_trace_id()

print(f"Trace ID: {trace3_id}")
print(f"Genre: {trace3_result['genre']}")
print(f"Songs found: {trace3_result['songs_found']}")
for song in trace3_result["top_songs"]:
    print(f"  {song['track_name']} by {song['artists']} (verified on Spotify: {song['verified_on_spotify']})")

##### Commentary
The initial search came back empty 3 times in a row before the fallback logic kicked in and returned 13 study genre tracks. The fallback did its job, the user never saw a blank result. But the judge correctly flagged the real gap: nothing in the pipeline actually verifies a track has no lyrics, "no lyrics" gets inferred from the genre label alone, not checked directly. That's worth noting as a known limitation rather than something to claim is solved.

#### Trace 4: Graceful Rejection 1 — Podcast Request

This trace tests whether the agent correctly recognizes a request that falls outside its scope and declines it cleanly instead of trying to force a music-based answer or hallucinating a response.

In [0]:
rejection_text_4 = run_rejection_trace(user_request_4)
trace4_id = mlflow.get_last_active_trace_id()

print(f"Trace ID: {trace4_id}")
print(f"User: {user_request_4}")
print(f"\nVibeML: {rejection_text_4}")

##### Commentary
Clean rejection. The agent didn't try to answer the podcast question or invent a recommendation, it declined directly and redirected back to playlists in one natural sentence.

#### Trace 5: Graceful Rejection 2 — Financial Advice Request

This trace tests a different kind of out-of-scope request, one that is not music-adjacent at all. This checks whether the agent holds its boundaries consistently across different types of irrelevant requests, not just ones that are loosely related to audio content like the podcast example.

In [0]:
rejection_text_5 = run_rejection_trace(user_request_5)
trace5_id = mlflow.get_last_active_trace_id()

print(f"Trace ID: {trace5_id}")
print(f"User: {user_request_5}")
print(f"\nVibeML: {rejection_text_5}")

##### Commentary
Same result as Trace 4, holds up even on a request with no music relation. The agent is not just pattern matching "non-music keywords," it is consistently staying in its lane across genuinely different types of off topic requests.

## Trace 6: Full Tool Coverage (Supplementary)

This trace is additional, it exists purely to demonstrate that every one of the 13 tools actually runs inside a real MLflow trace at least once - get_currently_playing, trend_filter, explicit_content_filter, feedback_refinement, generate_playlist_title_description, and save_playlist.

In [0]:
@mlflow.trace(name="vibeml_full_tool_coverage")
def run_full_coverage_trace(user_request):
    """
    Exercises every tool not already covered by Trace 1-5, so all 13 tools
    appear inside at least one real MLflow trace.
    """
    # Currently playing context
    currently_playing = get_currently_playing(sp)

    # Vibe mapping
    ranges = vibe_mapping(user_request, profile)
    explicit_genre = extract_explicit_genre(user_request)
    if explicit_genre:
        ranges["genre"] = explicit_genre

    # Song search
    results = search_songs(
        energy=(ranges["energy"]["min"], ranges["energy"]["max"]),
        valence=(ranges["valence"]["min"], ranges["valence"]["max"]),
        danceability=(ranges["danceability"]["min"], ranges["danceability"]["max"]),
        genre=ranges.get("genre"),
        limit=30,
        profile=profile,
        df_tracks=df_tracks,
        df_popularity=df_popularity
    )

    # Constraint filter
    results = constraint_filter(results, explicit=False, min_popularity=20)

    # Explicit content filter (redundant with constraint_filter, run here for coverage)
    results = explicit_content_filter(results, allow_explicit=False)

    # Trend filter
    results = trend_filter(results, popularity=(20, 100))

    # Deduplication
    results = deduplication(results, max_per_artist=1)

    # Liked songs matcher
    liked_matches = liked_songs_matcher(profile, ranges, df_tracks)
    if len(liked_matches) > 0:
        liked_matches["is_familiar"] = True
        results = pd.concat([liked_matches, results]).drop_duplicates(subset=["track_name"]).reset_index(drop=True)

    # Simulate a rejection so feedback_refinement actually runs
    tester_songs = results.head(3)
    rejected = [tester_songs.iloc[0]["track_name"]] if len(tester_songs) > 0 else []
    ranges = feedback_refinement([], rejected, ranges, results)

    # Re-search with refined ranges
    results = search_songs(
        energy=(ranges["energy"]["min"], ranges["energy"]["max"]),
        valence=(ranges["valence"]["min"], ranges["valence"]["max"]),
        danceability=(ranges["danceability"]["min"], ranges["danceability"]["max"]),
        genre=ranges.get("genre"),
        limit=20,
        profile=profile,
        df_tracks=df_tracks,
        df_popularity=df_popularity
    )
    results = constraint_filter(results, explicit=False, min_popularity=20)
    results = deduplication(results, max_per_artist=1)

    # Catalog verification
    top_results = results.head(5)
    verified_songs = []
    for _, row in top_results.iterrows():
        catalog_result = search_catalog(sp, track_name=row["track_name"], artist_name=row["artists"].split(";")[0])
        verified_songs.append({
            "track_name": row["track_name"],
            "artists": row["artists"],
            "track_genre": row["track_genre"],
            "verified_on_spotify": catalog_result is not None,
            "track_id": catalog_result["track_id"] if catalog_result else None
        })

    # Title and description
    sample_for_title = [{"track_name": s["track_name"], "artist": s["artists"].split(";")[0]} for s in verified_songs]
    playlist_meta = generate_playlist_title_description(user_request, sample_tracks=sample_for_title, profile=profile)

    # Save attempt
    track_ids = [s["track_id"] for s in verified_songs if s["track_id"]]
    save_result = save_playlist(sp, track_ids, playlist_meta["title"], playlist_meta["description"])

    return {
        "currently_playing": currently_playing,
        "genre": ranges.get("genre"),
        "rejected_then_refined": rejected,
        "songs_found": len(results),
        "top_songs": verified_songs,
        "playlist_title": playlist_meta["title"],
        "playlist_description": playlist_meta["description"],
        "save_result": save_result
    }

coverage_result = run_full_coverage_trace("I want something upbeat and popular for a party")
coverage_trace_id = mlflow.get_last_active_trace_id()

print(f"Trace ID: {coverage_trace_id}")
print(f"Currently playing: {coverage_result['currently_playing']}")
print(f"Genre: {coverage_result['genre']}")
print(f"Rejected then refined: {coverage_result['rejected_then_refined']}")
print(f"Songs found: {coverage_result['songs_found']}")
for song in coverage_result["top_songs"]:
    print(f"  {song['track_name']} by {song['artists']} (verified: {song['verified_on_spotify']})")
print(f"Title: {coverage_result['playlist_title']}")
print(f"Save result: {json.dumps(coverage_result['save_result'], indent=2)}")

##### Commentary

This trace confirms all 13 tools run correctly when chained together. Interestingly, "upbeat and popular for a party" resolved to a cluster of German party schlager tracks rather than anything else, which makes sense once you consider trend_filter was narrowing by popularity and the dataset's "party" genre tag happens to be dominated by that style. feedback_refinement also fired for real here after a simulated rejection, and the save attempt hit the same documented 403, consistent with every other save attempt in this notebook.

## Judge Evaluation

Runs vibe_match_judge against each of the four scoreable traces (Trace 1 GPT-OSS, Trace 1 Llama, Trace 2, Trace 3) and logs the result as Feedback directly onto each trace_id. Trace 4 and 5 are excluded since they are rejection examples, not song recommendations.

In [0]:
from mlflow.genai.judges import make_judge

vibe_match_judge = make_judge(
    name="vibe_match_quality",
    instructions=(
        "Rate how well the songs in {{ outputs }} match the user's request in {{ inputs }}. "
        "Score from 1 to 5, where 1 means the songs do not match at all and 5 means they are a strong match."
    ),
    model=f"databricks:/{LLM_MODEL}",
    feedback_value_type=int,
)

print(f"Judge created: {vibe_match_judge.name}")

In [0]:
from mlflow.entities import AssessmentSource, AssessmentSourceType

def score_and_log(trace_id, user_request, result):
    """
    Runs the judge on a trace's output and logs the score directly onto that trace_id.
    """
    songs_text = "\n".join([f"{s['track_name']} by {s['artists']}" for s in result["top_songs"]])
    
    judge_feedback = vibe_match_judge(
        inputs={"request": user_request},
        outputs={"songs": songs_text, "genre": result["genre"]}
    )
    
    mlflow.log_feedback(
        trace_id=trace_id,
        name=vibe_match_judge.name,
        value=judge_feedback.value,
        source=AssessmentSource(
            source_type=AssessmentSourceType.LLM_JUDGE,
            source_id=LLM_MODEL
        ),
        rationale=getattr(judge_feedback, "rationale", "")
    )
    
    return judge_feedback.value, getattr(judge_feedback, "rationale", "")

judge_records = []

score, rationale = score_and_log(trace1_id, user_request_1, trace1_result)
judge_records.append({"trace": "Trace 1 (GPT-OSS-120B)", "trace_id": trace1_id, "judge_score": score, "rationale": rationale})

score, rationale = score_and_log(trace1b_id, user_request_1, trace1b_result)
judge_records.append({"trace": "Trace 1 (Llama 3.3 70B)", "trace_id": trace1b_id, "judge_score": score, "rationale": rationale})

score, rationale = score_and_log(trace2_id, user_request_2, trace2_result)
judge_records.append({"trace": "Trace 2", "trace_id": trace2_id, "judge_score": score, "rationale": rationale})

score, rationale = score_and_log(trace3_id, user_request_3, trace3_result)
judge_records.append({"trace": "Trace 3", "trace_id": trace3_id, "judge_score": score, "rationale": rationale})

judge_df = pd.DataFrame(judge_records)

for _, row in judge_df.iterrows():
    print(f"{row['trace']} (score: {row['judge_score']})")
    print(f"  {row['rationale']}\n")

##### Commentary
Scores came back 2, 3, 4, 3. The standout result is GPT-OSS-120B's 2 on Trace 1, the lowest score either model has gotten on this request across runs, caused by genuinely off-genre picks like Plastilina Mosh and Criolo getting pulled into an "afrobeat" search. Llama held steady at 3 with songs that are at least all tagged correctly. Trace 2 again scored highest since there was no specific qualifier to miss. Trace 3's 3 repeats the same known gap, lyric-free is assumed from genre, never directly verified.

### Formal Evaluation Run

This re-scores the same 4 traces using mlflow.genai.evaluate(), which creates a proper Evaluation Run in MLflow with the inline results link.

In [0]:
all_traces_df = mlflow.search_traces(max_results=20)
print(all_traces_df.columns.tolist())
all_traces_df.head()

target_ids = [trace1_id, trace1b_id, trace2_id, trace3_id]

traces_df = all_traces_df[all_traces_df["trace_id"].isin(target_ids)]
print(f"Found {len(traces_df)} of 4 target traces")

eval_results = mlflow.genai.evaluate(
    data=traces_df,
    scorers=[vibe_match_judge],
)

## Human-in-the-Loop Evaluation

We review the same four traces the judge scored and logs its own rating directly onto each trace_id, using the same feedback name (vibe_match_quality) as the judge so both scores sit side by side on the same trace in the MLflow UI.

In [0]:
team_reviews = [
    {
        "trace": "Trace 1 (GPT-OSS-120B)",
        "trace_id": trace1_id,
        "team_score": 1,
        "team_rationale": "Enzo and Fellini aren't afrobeat at all, they're Mexican rock and Brazilian rap. Only the two Limoblaze tracks are genuinely gospel afrobeat adjacent. This doesn't hold up as a fast gym afrobeat list."
    },
    {
        "trace": "Trace 1 (Llama 3.3 70B)",
        "trace_id": trace1b_id,
        "team_score": 3,
        "team_rationale": "All three tracks are legitimately tagged afrobeat. Soul Makossa is a classic but on the slower side for a gym session, otherwise solid."
    },
    {
        "trace": "Trace 2",
        "trace_id": trace2_id,
        "team_score": 4,
        "team_rationale": "Deep-house picks genuinely fit a late night drive, and leading with a song literally called Drive is a nice touch."
    },
    {
        "trace": "Trace 3",
        "trace_id": trace3_id,
        "team_score": 3,
        "team_rationale": "Genre and mood read right, but agree with the judge that lyric-free isn't actually confirmed anywhere, just assumed from the genre label."
    }
]

for review in team_reviews:
    mlflow.log_feedback(
        trace_id=review["trace_id"],
        name=vibe_match_judge.name,
        value=review["team_score"],
        source=AssessmentSource(
            source_type=AssessmentSourceType.HUMAN,
            source_id="vibeml_team"
        ),
        rationale=review["team_rationale"]
    )

team_df = pd.DataFrame(team_reviews)

comparison_df = judge_df[["trace", "trace_id", "judge_score"]].merge(
    team_df[["trace", "team_score", "team_rationale"]], on="trace"
)
comparison_df["agreement"] = comparison_df["judge_score"] == comparison_df["team_score"]

comparison_df

### Commentary: Human-in-the-Loop Review

The team disagreed with the judge on Trace 1 GPT-OSS-120B specifically, scoring it a 1 against the judge's 2. Looking closer at the actual songs, Enzo and Fellini are not afrobeat by any reasonable definition, they're Mexican rock and Brazilian rap that slipped through the genre filter. The judge's 2 already flagged the problem, the team's review just found it even less defensible on a closer listen. This matches a pattern that's shown up before, when a genre specific request pulls in mistagged tracks, we tend to score it more harshly than the judge does. The team agreed exactly with the judge on the other 3 traces, including Llama's version of this same request, which had no mismatch.

## Overall Agent Performance Summary

This section ties together what the five traces and the evaluation showed about how the agent performs overall, and compares the two LLMs directly.

### Final Commentary

The clearest finding this run is that the two LLMs are not reliably equivalent. In an earlier run they converged on identical songs and tied scores, this run they diverged, with GPT-OSS-120B pulling in genuinely mistagged tracks that dragged its score down to a 2, while Llama 3.3 70B stayed accurate at 3. That inconsistency is itself the finding, model choice can matter even when the rest of the pipeline, genre detection, audio feature filtering, stays exactly the same. Beyond that, the agent continues to handle mood based requests well, Trace 2's deep-house result scored highest both runs, and continues to stay correctly in scope on off topic requests. The recurring weak spot is verifying specific qualifiers inside a request, fast tempo, no lyrics, rather than just the genre label.